# SAPT Analysis Workflow (Psi4, Colab) — Gaussian `.log` → Excel

Simplified, batch-capable workflow:

1. Install Psi4
2. Upload one **or more** Gaussian `.log` files
3. Auto-extract the last (optimized) geometry from each
4. Define the two fragments (applied to every uploaded file)
5. Set the SAPT method/basis
6. Auto-generate each Psi4 input file
7. Run Psi4 for every file
8. Parse SAPT results
9. Export **all results into one Excel file**

> Only **Part 4** (fragments) and **Part 5** (SAPT settings) normally need edits.
> Fragment definition and SAPT settings are applied to every uploaded `.log` file, so this
> works best when all uploaded files are the same dimer/complex (e.g. different conformers,
> optimization steps, or scan points of the same two fragments).


## Part 1 — Install Psi4

Installs micromamba and uses it to install Psi4 from `conda-forge` in an isolated environment.


In [1]:
#@title 1.1 Install Micromamba
import os, subprocess, sys, pathlib, time

MAMBA_ROOT = "/content/micromamba"
ENV_NAME = "psi4env"
os.environ["MAMBA_ROOT_PREFIX"] = MAMBA_ROOT

def run(cmd, **kw):
    print(f"$ {cmd}")
    result = subprocess.run(cmd, shell=True, text=True, capture_output=True, **kw)
    if result.stdout:
        print(result.stdout[-4000:])
    if result.returncode != 0:
        print(result.stderr[-4000:])
        raise RuntimeError(f"Command failed ({result.returncode}): {cmd}")
    return result

if not pathlib.Path("/content/bin/micromamba").exists():
    print("Downloading micromamba...")
    run("mkdir -p /content/bin")
    run(
        "curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest "
        "| tar -xvj -C /content/bin --strip-components=1 bin/micromamba"
    )
else:
    print("micromamba already installed.")

MICROMAMBA = "/content/bin/micromamba"
run(f"{MICROMAMBA} --version")
print("Micromamba installed at", MICROMAMBA)


$ mkdir -p /content/bin
$ curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xvj -C /content/bin --strip-components=1 bin/micromamba
bin/micromamba

$ /content/bin/micromamba --version
2.8.1

Micromamba installed at /content/bin/micromamba


In [2]:
#@title 1.2 Install Psi4 (creates the 'psi4env' environment)
start = time.time()

env_exists = subprocess.run(
    f"{MICROMAMBA} env list", shell=True, text=True, capture_output=True
).stdout
if ENV_NAME not in env_exists:
    print(f"Creating environment '{ENV_NAME}' and installing Psi4 (this may take 5-15 min)...")
    run(
        f"{MICROMAMBA} create -y -n {ENV_NAME} -c conda-forge "
        f"psi4 numpy pandas openpyxl python=3.10"
    )
else:
    print(f"Environment '{ENV_NAME}' already exists, skipping install.")

elapsed = time.time() - start
print(f"Done. Elapsed: {elapsed/60:.1f} min")

PSI4_PYTHON = f"{MAMBA_ROOT}/envs/{ENV_NAME}/bin/python"
PSI4_BIN = f"{MAMBA_ROOT}/envs/{ENV_NAME}/bin/psi4"
print("Psi4 python:", PSI4_PYTHON)
print("Psi4 binary:", PSI4_BIN)


Creating environment 'psi4env' and installing Psi4 (this may take 5-15 min)...
$ /content/bin/micromamba create -y -n psi4env -c conda-forge psi4 numpy pandas openpyxl python=3.10
ex-4.5-7_kmp_llvm
Linking zstd-1.5.7-hb78ec9c_6
Linking libgcc-15.2.0-he0feb66_19
Linking ld_impl_linux-64-2.45.1-default_hbd61a6d_102
Linking libdwarf-2.1.0-h43af8da_1
Linking libffi-3.5.2-h3435931_0
Linking libsqlite-3.53.3-h0c1763c_0
Linking tk-8.6.13-noxft_h366c992_103
Linking libiconv-1.18-h3b78370_2
Linking keyutils-1.6.3-hb9d3cd8_0
Linking yaml-0.2.5-h280c20c_3
Linking bzip2-1.0.8-hda65f42_9
Linking libnsl-2.0.1-hb9d3cd8_1
Linking libexpat-2.8.1-hecca717_1
Linking libuuid-2.42.2-h5347b49_0
Linking libgcc-ng-15.2.0-h69a702a_19
Linking c-ares-1.34.6-hb03c661_0
Linking libgfortran5-15.2.0-h68bc16d_19
Linking libxc-c-7.0.0-cpu_h75f4f88_6
Linking libstdcxx-15.2.0-h934c35e_19
Linking liblzma-5.8.3-hb03c661_0
Linking ncurses-6.6-hdb14827_0
Linking gau2grid-2.0.9-hb03c661_0
Linking libxcrypt-4.4.36-hd590300_1


In [3]:
#@title 1.3 Verify Installation
verify = subprocess.run(
    f'{PSI4_PYTHON} -c "import psi4; print(psi4.__version__)"',
    shell=True, text=True, capture_output=True
)
if verify.returncode == 0:
    print(f"\u2713 Psi4 installed successfully. Version: {verify.stdout.strip()}")
else:
    print("\u2717 Psi4 verification failed:")
    print(verify.stderr)
    raise RuntimeError("Psi4 is not importable in the psi4env environment.")


✓ Psi4 installed successfully. Version: 1.11


## Part 2 — Upload File(s)

Select **one or more** Gaussian `.log` files. Each one will get its own SAPT run; all results
land in a single Excel file at the end (Part 9).


In [4]:
#@title 2.1 Upload
from google.colab import files

UPLOAD_DIR = "/content/uploads"
os.makedirs(UPLOAD_DIR, exist_ok=True)

print("Select one or more Gaussian .log files (multi-select is supported).")
uploaded = files.upload()

uploaded_paths = []
for fname, content in uploaded.items():
    if pathlib.Path(fname).suffix.lower() != ".log":
        raise ValueError(f"Unsupported file: {fname}. Only .log files are accepted here.")
    dest = os.path.join(UPLOAD_DIR, fname)
    with open(dest, "wb") as f:
        f.write(content)
    uploaded_paths.append(dest)
    print(f"Saved: {dest} ({len(content)/1024:.1f} KB)")

print(f"\n{len(uploaded_paths)} file(s) ready for processing:")
for p in uploaded_paths:
    print(" -", p)


Select one or more Gaussian .log files (multi-select is supported).


Saving 1.log to 1.log
Saved: /content/uploads/1.log (557.8 KB)

1 file(s) ready for processing:
 - /content/uploads/1.log


## Part 3 — Automatic Geometry Extraction

For each `.log` file: finds the **last** optimized geometry (final "Standard orientation" /
"Input orientation" block), counts atoms, detects atomic symbols, builds Cartesian coordinates.


In [5]:
#@title 3.1 Geometry extraction functions
import re

PERIODIC_TABLE = {
    1:"H",2:"He",3:"Li",4:"Be",5:"B",6:"C",7:"N",8:"O",9:"F",10:"Ne",
    11:"Na",12:"Mg",13:"Al",14:"Si",15:"P",16:"S",17:"Cl",18:"Ar",
    19:"K",20:"Ca",21:"Sc",22:"Ti",23:"V",24:"Cr",25:"Mn",26:"Fe",27:"Co",
    28:"Ni",29:"Cu",30:"Zn",31:"Ga",32:"Ge",33:"As",34:"Se",35:"Br",36:"Kr",
    37:"Rb",38:"Sr",39:"Y",40:"Zr",41:"Nb",42:"Mo",43:"Tc",44:"Ru",45:"Rh",
    46:"Pd",47:"Ag",48:"Cd",49:"In",50:"Sn",51:"Sb",52:"Te",53:"I",54:"Xe",
    55:"Cs",56:"Ba",57:"La",78:"Pt",79:"Au",80:"Hg",82:"Pb",
}

def _parse_gaussian_orientation_blocks(text):
    """Return list of geometries (each: list of (Z, x, y, z)) found in a Gaussian
    .log file, in file order.

    Gaussian prints orientation blocks with THREE dashed separator lines:
        <Title>:
        ---------------------------------------------------------------------
         Center     Atomic      Atomic             Coordinates (Angstroms)
         Number     Number       Type              X           Y           Z
        ---------------------------------------------------------------------
            1          6             0       -1.234567    0.123456    0.000000
            ...
        ---------------------------------------------------------------------
    i.e. dashes -> 2 header lines -> dashes -> DATA -> dashes. The coordinate
    data sits between the 2nd and 3rd dashed line, not the 1st and 2nd.
    """
    blocks = []
    titles = re.compile(r"(Standard orientation|Input orientation|Z-Matrix orientation):")
    for m in titles.finditer(text):
        window = text[m.end(): m.end() + 4000]
        dashes = list(re.finditer(r"-{5,}", window))
        if len(dashes) < 3:
            continue
        data_start = dashes[1].end()
        data_end = dashes[2].start()
        body = window[data_start:data_end]
        rows = []
        for line in body.strip("\n").split("\n"):
            parts = line.split()
            if len(parts) >= 6 and parts[0].isdigit() and parts[1].isdigit():
                atomic_num = int(parts[1])
                try:
                    x, y, z = map(float, parts[-3:])
                except ValueError:
                    continue
                rows.append((atomic_num, x, y, z))
        if rows:
            blocks.append(rows)
    return blocks

def extract_last_geometry(path):
    """Returns list of (symbol, x, y, z) for the LAST geometry in a .log file."""
    text = pathlib.Path(path).read_text(errors="ignore")
    blocks = _parse_gaussian_orientation_blocks(text)
    if not blocks:
        raise ValueError(
            f"No geometry blocks found in {path}. This parser looks for Gaussian "
            "'Standard orientation' / 'Input orientation' / 'Z-Matrix orientation' "
            "blocks. If your file uses a different print format (e.g. from a "
            "different program, or a truncated/partial log), share a snippet and "
            "the parser can be extended."
        )
    last = blocks[-1]
    rows = [(PERIODIC_TABLE.get(z, str(z)), x, y, z_) for (z, x, y, z_) in last]
    if not rows:
        raise ValueError(f"Could not extract any atoms from {path}")
    return rows

def geometry_to_cartesian_block(rows):
    return "\n".join(f"{s:<3}{x:>15.8f}{y:>15.8f}{z:>15.8f}" for s, x, y, z in rows)

print("Geometry-extraction functions ready.")


Geometry-extraction functions ready.


In [6]:
#@title 3.2 Extract geometry from every uploaded file
extracted = {}   # path -> list of (symbol, x, y, z)

for p in uploaded_paths:
    rows = extract_last_geometry(p)
    extracted[p] = rows
    print(f"{os.path.basename(p)}: {len(rows)} atoms, "
          f"elements={sorted(set(r[0] for r in rows))}")

print(f"\n\u2713 Extracted geometry from {len(extracted)} file(s).")


1.log: 40 atoms, elements=['C', 'H', 'N', 'O']

✓ Extracted geometry from 1 file(s).


## Part 4 — Fragment Definition

**Fragments are detected automatically** from the optimized geometry's bonding pattern
(atoms within covalent-bonding distance of each other = same fragment). This correctly
handles cases where the two fragments are **not** a simple "first N atoms / rest" split —
e.g. a small guest atom that appears in the middle of the host's atom list.

If your file legitimately has more than 2 disconnected pieces, or you want to force a
specific split, set `fragment1_atoms` / `fragment2_atoms` to explicit 1-based atom index
lists instead of leaving them as `None`.


In [7]:
#@title 4.1 Fragment detection (auto) \u2014 edit only if you need a manual override
# Leave both as None to auto-detect fragments from bonding distances.
# To force a specific split, set explicit 1-based atom index lists, e.g.:
#   fragment1_atoms = [1,2,3,4,5,6,7,8,9,10]
#   fragment2_atoms = [11,12,13,14]
fragment1_atoms = None
fragment2_atoms = None

charge1 = 0
mult1   = 1

charge2 = 0
mult2   = 1

# Approximate covalent radii (Angstrom) for common elements; unlisted elements
# fall back to DEFAULT_RADIUS.
COV_RADIUS = {
    "H": 0.31, "C": 0.76, "N": 0.71, "O": 0.66, "F": 0.57,
    "P": 1.07, "S": 1.05, "Cl": 0.99, "Br": 1.14, "I": 1.33,
}
DEFAULT_RADIUS = 0.77
BOND_SCALE = 1.3  # tolerance multiplier on the sum of covalent radii

def detect_fragments(rows):
    """Split a geometry into disconnected fragments via a covalent-radius bond
    graph. Returns a list of fragments, each a list of 0-based atom indices,
    in the order first encountered."""
    n = len(rows)
    adj = {i: set() for i in range(n)}
    for i in range(n):
        si, xi, yi, zi = rows[i]
        for j in range(i + 1, n):
            sj, xj, yj, zj = rows[j]
            d = ((xi - xj) ** 2 + (yi - yj) ** 2 + (zi - zj) ** 2) ** 0.5
            thresh = (COV_RADIUS.get(si, DEFAULT_RADIUS) + COV_RADIUS.get(sj, DEFAULT_RADIUS)) * BOND_SCALE
            if d < thresh:
                adj[i].add(j)
                adj[j].add(i)
    seen, comps = set(), []
    for i in range(n):
        if i in seen:
            continue
        stack, comp = [i], set()
        while stack:
            c = stack.pop()
            if c in seen:
                continue
            seen.add(c)
            comp.add(c)
            stack.extend(adj[c] - seen)
        comps.append(sorted(comp))
    return comps

fragments = {}  # path -> (frag1_rows, frag2_rows, frag1_idx_1based, frag2_idx_1based)

for p, rows in extracted.items():
    stem = os.path.basename(p)
    if fragment1_atoms is not None and fragment2_atoms is not None:
        f1_idx = [a - 1 for a in fragment1_atoms]
        f2_idx = [a - 1 for a in fragment2_atoms]
    else:
        comps = detect_fragments(rows)
        if len(comps) != 2:
            detected = [[c + 1 for c in comp] for comp in comps]
            raise ValueError(
                f"Automatic fragment detection found {len(comps)} disconnected piece(s) "
                f"in '{stem}', not 2. Detected pieces (1-based indices): {detected}. "
                "Set fragment1_atoms / fragment2_atoms manually above to force the split "
                "you want (e.g. merge pieces that belong together, or exclude solvent)."
            )
        f1_idx, f2_idx = comps[0], comps[1]

    if not f1_idx or not f2_idx:
        raise ValueError(f"One of the fragments is empty for '{stem}'.")

    f1_rows = [rows[i] for i in f1_idx]
    f2_rows = [rows[i] for i in f2_idx]
    fragments[p] = (f1_rows, f2_rows, [i + 1 for i in f1_idx], [i + 1 for i in f2_idx])

for p, (f1, f2, f1_idx, f2_idx) in fragments.items():
    print(f"{os.path.basename(p)}:")
    print(f"  fragment1: {len(f1)} atoms -> 1-based indices {f1_idx}")
    print(f"  fragment2: {len(f2)} atoms -> 1-based indices {f2_idx}")

print(f"\n\u2713 Two fragments defined for every file "
      f"({'auto-detected from bonding' if fragment1_atoms is None else 'manual override'}).")


1.log:
  fragment1: 36 atoms -> 1-based indices [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 34, 35, 36, 37]
  fragment2: 4 atoms -> 1-based indices [33, 38, 39, 40]

✓ Two fragments defined for every file (auto-detected from bonding).


## Part 5 — SAPT Settings

Applied to every uploaded file. Change `method` to switch SAPT levels
(`sapt0`, `sapt2`, `sapt2+`, `sapt2+(3)`, ...) without touching any other code.


In [8]:
#@title 5.1 EDIT THESE VALUES
method   = "sapt0"        # e.g. sapt0, sapt2, "sapt2+", "sapt2+(3)"
basis    = "jun-cc-pvdz"
memory   = "8 GB"
nthreads = 2

print(f"Method   : {method}")
print(f"Basis    : {basis}")
print(f"Memory   : {memory}")
print(f"Threads  : {nthreads}")


Method   : sapt0
Basis    : jun-cc-pvdz
Memory   : 8 GB
Threads  : 2


## Part 6 — Generate Psi4 Input

Builds a `sapt.in` for every uploaded file, each in its own run folder named after the log file.


In [9]:
#@title 6.1 Build sapt.in for every file
RUNS_DIR = "/content/sapt_runs"
os.makedirs(RUNS_DIR, exist_ok=True)

def frag_block(rows):
    return "\n".join(f"{s:<3}{x:>15.8f}{y:>15.8f}{z:>15.8f}" for s, x, y, z in rows)

run_info = {}  # path -> dict(workdir, sapt_in, sapt_out, stem)

for p, (f1, f2, f1_idx, f2_idx) in fragments.items():
    stem = pathlib.Path(p).stem
    workdir = os.path.join(RUNS_DIR, stem)
    os.makedirs(workdir, exist_ok=True)

    molecule_block = f"""
{charge1} {mult1}
{frag_block(f1)}
--
{charge2} {mult2}
{frag_block(f2)}
"""

    psi4_input = f"""# Auto-generated SAPT input for {stem}
memory {memory}

molecule dimer {{
{molecule_block}
units angstrom
no_reorient
no_com
}}

set {{
    basis {basis}
    scf_type df
    freeze_core true
}}

energy('{method}')
"""

    sapt_in_path = os.path.join(workdir, "sapt.in")
    with open(sapt_in_path, "w") as f:
        f.write(psi4_input)

    run_info[p] = {
        "stem": stem,
        "workdir": workdir,
        "sapt_in": sapt_in_path,
        "sapt_out": os.path.join(workdir, "sapt.out"),
    }
    print(f"Wrote {sapt_in_path}")

print(f"\n\u2713 Generated {len(run_info)} Psi4 input file(s).")


Wrote /content/sapt_runs/1/sapt.in

✓ Generated 1 Psi4 input file(s).


## Part 7 — Run Psi4

Runs every generated input in turn, reporting progress, elapsed time, and errors per file.
A failure on one file is reported but does not stop the others.


In [10]:
#@title 7.1 Run all SAPT calculations
failed = []
succeeded = []

for i, (p, info) in enumerate(run_info.items(), 1):
    stem = info["stem"]
    print(f"\n[{i}/{len(run_info)}] Running {stem} ({method}/{basis})...")
    t0 = time.time()

    proc = subprocess.Popen(
        [PSI4_BIN, "-i", info["sapt_in"], "-o", info["sapt_out"], "-n", str(nthreads)],
        cwd=info["workdir"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
    )
    while proc.poll() is None:
        elapsed = time.time() - t0
        print(f"\r  running... elapsed {elapsed:6.1f}s", end="", flush=True)
        time.sleep(5)
    print()

    returncode = proc.wait()
    elapsed = time.time() - t0

    out_text = ""
    if os.path.exists(info["sapt_out"]):
        out_text = pathlib.Path(info["sapt_out"]).read_text(errors="ignore")

    if returncode != 0:
        print(f"  \u2717 {stem} FAILED after {elapsed:.1f}s")
        error_lines = [l.strip() for l in out_text.split("\n") if "error" in l.lower()]
        for e in error_lines[:10]:
            print("    ", e)
        failed.append(stem)
    else:
        print(f"  \u2713 {stem} finished in {elapsed:.1f}s ({elapsed/60:.1f} min)")
        succeeded.append(stem)

print(f"\n{'='*50}")
print(f"Completed: {len(succeeded)} succeeded, {len(failed)} failed")
if failed:
    print("Failed runs:", failed)



[1/1] Running 1 (sapt0/jun-cc-pvdz)...
  running... elapsed 1610.9s
  ✓ 1 finished in 1615.9s (26.9 min)

Completed: 1 succeeded, 0 failed


## Part 8 — Parse Results

Extracts Electrostatics, Exchange, Induction, Dispersion, and Total SAPT for every successful
run, in Hartree, kcal/mol, and kJ/mol.


In [14]:
#@title 8.1 Parsing functions and unit conversion
HARTREE_TO_KCAL = 627.509474
HARTREE_TO_KJ   = 2625.499639

SAPT_LABELS = {
    "Electrostatics": r"Electrostatics\s+([-\d.]+)\s*\[mEh\]",
    "Exchange":       r"Exchange\s+([-\d.]+)\s*\[mEh\]",
    "Induction":      r"Induction\s+([-\d.]+)\s*\[mEh\]",
    "Dispersion":     r"Dispersion\s+([-\d.]+)\s*\[mEh\]",
    "Total SAPT":     r"Total\s+SAPT\s*\S*\s+([-\d.]+)\s*\[mEh\]",
}

def parse_sapt_output(text):
    results = {}
    for label, pattern in SAPT_LABELS.items():
        matches = re.findall(pattern, text)
        if matches:
            results[label] = float(matches[-1])
    missing = [k for k in SAPT_LABELS if k not in results]
    if missing:
        raise ValueError(f"Could not find these SAPT components: {missing}")
    return results

print("Parsing functions ready.")


Parsing functions ready.


In [15]:
#@title 8.2 Parse every successful run
import pandas as pd

all_results = {}   # stem -> dict(component -> hartree)
parse_failed = []

for p, info in run_info.items():
    stem = info["stem"]
    if stem in failed:
        continue
    try:
        text = pathlib.Path(info["sapt_out"]).read_text(errors="ignore")
        all_results[stem] = parse_sapt_output(text)
        total_kcal = all_results[stem]["Total SAPT"] * HARTREE_TO_KCAL
        print(f"{stem}: Total SAPT = {total_kcal:.3f} kcal/mol")
    except ValueError as e:
        print(f"\u2717 Could not parse {stem}: {e}")
        parse_failed.append(stem)

print(f"\n\u2713 Parsed {len(all_results)} file(s).")
if parse_failed:
    print("Parse failures:", parse_failed)


1: Total SAPT = -12399.832 kcal/mol

✓ Parsed 1 file(s).


## Part 9 — Save Results (Excel)

Writes **one Excel file** with:
- a **Summary** sheet (one row per uploaded file, Total SAPT + all components in kcal/mol)
- one **detail sheet per file** (full decomposition in Hartree / kcal/mol / kJ/mol)


In [16]:
#@title 9.1 Build and download SAPT_Results.xlsx
OUT_DIR = "/content/sapt_results"
os.makedirs(OUT_DIR, exist_ok=True)
xlsx_path = os.path.join(OUT_DIR, "SAPT_Results.xlsx")

# --- Summary sheet ---------------------------------------------------------
summary_rows = []
for stem, comps in all_results.items():
    row = {"File": stem, "Method": method, "Basis": basis}
    for label, meh in comps.items():
        row[f"{label} (kcal/mol)"] = round(meh * HARTREE_TO_KCAL, 4)
    summary_rows.append(row)
summary_df = pd.DataFrame(summary_rows)

with pd.ExcelWriter(xlsx_path, engine="openpyxl") as writer:
    summary_df.to_excel(writer, sheet_name="Summary", index=False)

    for stem, comps in all_results.items():
        detail_rows = [
            {
                "Component": label,
                "Hartree": meh,
                "kcal/mol": meh * HARTREE_TO_KCAL,
                "kJ/mol": meh * HARTREE_TO_KJ,
            }
            for label, meh in comps.items()
        ]
        detail_df = pd.DataFrame(detail_rows)
        # Excel sheet names max 31 chars
        sheet_name = stem[:31]
        detail_df.to_excel(writer, sheet_name=sheet_name, index=False)

print(f"Saved: {xlsx_path}")
print(f"Sheets: Summary + {len(all_results)} per-file detail sheet(s)")
if failed or parse_failed:
    print("\nNote: the following files are NOT in the Excel output "
          f"(run or parse failed): {sorted(set(failed) | set(parse_failed))}")

files.download(xlsx_path)


Saved: /content/sapt_results/SAPT_Results.xlsx
Sheets: Summary + 1 per-file detail sheet(s)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

---
## Done

`SAPT_Results.xlsx` contains a **Summary** sheet comparing all uploaded files side by side, plus
a full energy-decomposition sheet for each one.

To add more files later, just re-run from **Part 2** with new uploads (or add to
`uploaded_paths`) — Parts 3-9 will pick them up automatically.
